# defining zone

In [ ]:
import _bootstrap  # noqa: F401  — puts src/ on sys.path (see src/analysis/README.md)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
import params
import pycountry as pc
import requests
import json
import time
import random
import os
import joblib
from matplotlib import pyplot
pyplot.rcParams['figure.dpi'] = 300
pyplot.rcParams['savefig.dpi'] = 600


dropbox_path = params.dropbox_path.parents[1]/'adolescent_modelling'
constructed_ages = [[25,49],[15,20],[21,49],[15,24],[25,34],[35,49]]


def save_model(model, file_path):
    joblib.dump(model, file_path)

def load_model(file_path):
    return joblib.load(file_path) if os.path.exists(file_path) else None

# fb data preparation 

In [ ]:
# prepare fb data 
from datetime import datetime
import os 
import pycountry as pc
 

def read_all_fb_national_files():
    """
    Function to read all the fb_national files
    (from 2015 to the latest file in averaged/national and return the combined data)
    it also reads the unpop file and append the latest year to the current year and merge with the fb_national file
    """


    # read the fb_national file
    df_fb_national = pd.read_csv(params.dropbox_path.parents[2] / 'pipeline/pre_pipeline_result/fb_national.csv')
    df_fb_national.rename(columns={x: x.lower() for x in df_fb_national.columns}, inplace=True)
    df_fb_national['timestamp'] = pd.to_datetime([x.replace('.0', '') for x in df_fb_national['year'].astype(str) + '-' + df_fb_national['month'].astype(str) + '-01'])
    max_time = df_fb_national['timestamp'].max()

    files = os.listdir(params.dropbox_path.parents[2] / 'pipeline/averaged/national')
    times = [datetime.strptime(x.replace('upper_','').split('_')[1],'%Y%m')  for x in files if '.csv' in x]
    for time in times:
        if time > max_time:
            #print(f'processing {time} using the unpop from {max_time}')
            df = pd.read_csv(params.dropbox_path.parents[2] / f'pipeline/averaged/national/mau_upper_{time.strftime("%Y%m")}_averaged.csv')
            df['timestamp'] = time

            df.rename(columns={x: x.lower() for x in df.columns}, inplace=True)
            df['country']=df['country'].fillna('NA')
            df['iso3'] = [None if pd.isnull(x) else pc.countries.get(alpha_2=x).alpha_3 if x not in ['AN', 'XK'] else {'AN': 'ANT', 'XK': 'XKX'}[x] for x in df['country']]
            df_fb_national = pd.concat([df_fb_national,df.drop(columns='country')], axis=0)

    df_fb_national.sort_values(by=['iso3','year','month'], inplace=True)
    df_fb_national.reset_index(drop=True, inplace=True)

    

    return df_fb_national


## fb - standardise with the unpop data

In [ ]:
from datetime import datetime

# Read UN population data
df_unpop = pd.read_csv(dropbox_path/'data/file/un_1950_2023_processed.csv') 
df_unpop.columns = [x.lower() for x in df_unpop.columns]
df_unpop['year']=df_unpop['year'].astype(int)

# Append year up to current year to the current unpop file
max_year = df_unpop['year'].max()
current_year = datetime.now().year
if current_year > max_year:
    for year in range(max_year + 1, current_year + 1):
        temp = df_unpop.loc[df_unpop['year'] == max_year,].copy()
        temp['year'] = year
        df_unpop = pd.concat([df_unpop, temp], axis=0)
    df_unpop.sort_values(by=['iso3', 'year'], inplace=True)
    df_unpop.reset_index(drop=True, inplace=True)

df_unpop.rename(columns={x: x.lower() for x in df_unpop.columns}, inplace=True)


In [ ]:
# Read all Facebook national files and combine them into a single DataFrame
df_fb_all = read_all_fb_national_files()

# Select specific columns from the combined Facebook data
df_selected = df_fb_all[['year', 'month', 'iso3', 'audience_type', 'fb_all',
    'fb_age_18_plus_men', 'fb_age_18_plus_women', 'fb_age_18_plus_ratio',
    'fb_age_25_49_men', 'fb_age_25_49_women', 'fb_age_25_49_ratio']].copy()

# Generate additional columns for specific age groups by summing relevant columns
for col_type in ['men', 'women']:
    # Age group 15-20
    df_selected[f'fb_age_15_20_{col_type}'] = df_fb_all[f'fb_age_15_19_{col_type}']
    # Age group 15-24
    df_selected[f'fb_age_15_24_{col_type}'] = df_fb_all[f'fb_age_15_19_{col_type}'] + df_fb_all[f'fb_age_20_24_{col_type}']
    # Age group 25-34
    df_selected[f'fb_age_25_34_{col_type}'] = df_fb_all[f'fb_age_25_29_{col_type}'] + df_fb_all[f'fb_age_30_34_{col_type}']
    # Age group 35-49
    df_selected[f'fb_age_35_49_{col_type}'] = df_fb_all[f'fb_age_35_39_{col_type}'] + df_fb_all[f'fb_age_40_44_{col_type}'] + df_fb_all[f'fb_age_45_49_{col_type}']
    # Age group 21-49
    df_selected[f'fb_age_21_49_{col_type}'] = df_fb_all[f'fb_age_20_24_{col_type}'] + df_selected[f'fb_age_25_34_{col_type}'] + df_selected[f'fb_age_35_49_{col_type}']

# Rename columns to a standardized format
df_selected.columns = [x.replace('age_', '').replace("plus", "999").replace("women", "f").replace("ratio", "r").replace("men", "m") for x in df_selected.columns]

# Group the data by 'iso3' and 'year', calculate the mean for each group, and reset the index
df_selected_fb_grouped = df_selected.drop(columns=['audience_type']).groupby(['iso3', 'year']).mean().reset_index()

# Display the unique years in the grouped data
df_selected_fb_grouped['year'].unique()

In [ ]:
# standardise the columns
df_all = pd.merge(df_selected_fb_grouped,df_unpop, on=['iso3','year'],how='left')

fb_col, pop_col = 'fb_18_999_r','18_inf_r'
male_col,female_col = pop_col.replace("_r",'_m'),pop_col.replace("_r",'_f')
fb_male_col,fb_female_col = fb_col.replace("_r",'_m'),fb_col.replace("_r",'_f')

#df_all[f'fb_18_999_ratio'] = df_all[fb_col] / df_all[pop_col]
df_all[f'fb_18_999_wom'] = df_all[fb_female_col]/(df_all[female_col])
df_all[f'fb_18_999_men'] = df_all[fb_male_col] / (df_all[male_col])
df_all[f'fb_18_999_r'] = df_all[f'fb_18_999_wom']/df_all[f'fb_18_999_men']

# all age groups 
# firstly generating the columns for the age groups (for both the fb and the population) 
for col_type in ['f','m']:
    for suffix in ['']:
            
        df_all[f'{suffix}15_20_{col_type}'] = df_all[f'{suffix}15_19_{col_type}']
        df_all[f'{suffix}15_24_{col_type}'] = df_all[f'{suffix}15_19_{col_type}']+df_all[f'{suffix}20_24_{col_type}']
        df_all[f'{suffix}25_34_{col_type}'] = df_all[f'{suffix}25_29_{col_type}']+df_all[f'{suffix}30_34_{col_type}']
        df_all[f'{suffix}35_49_{col_type}'] = df_all[f'{suffix}35_39_{col_type}']+df_all[f'{suffix}40_44_{col_type}']+df_all[f'{suffix}45_49_{col_type}']
        df_all[f'{suffix}21_49_{col_type}'] = df_all[f'{suffix}20_24_{col_type}']+df_all[f'{suffix}25_34_{col_type}'] +df_all[f'{suffix}35_49_{col_type}'] 
    
# all other age groups that we will use 
constructed_ages = [[25,49],[15,20],[21,49],[15,24],[25,34],[35,49]]
for age in constructed_ages:
    
    age_min,age_max = age[0],age[1]
    fb_col, pop_col = f'fb_{age_min}_{age_max}_r',f'{age_min}_{age_max}_r'
    male_col,female_col = pop_col.replace("_r",'_m'),pop_col.replace("_r",'_f')
    fb_male_col,fb_female_col = fb_col.replace("_r",'_m'),fb_col.replace("_r",'_f')
        
    
    #df_all[f'fb_{age_min}_{age_max}_ratio'] = df_all[fb_col] / df_all[pop_col]
    df_all[f'fb_{age_min}_{age_max}_wom'] = df_all[fb_female_col]/(df_all[female_col])
    df_all[f'fb_{age_min}_{age_max}_men'] = df_all[fb_male_col] /(df_all[male_col])
    df_all[f'fb_{age_min}_{age_max}_r'] = df_all[f'fb_{age_min}_{age_max}_wom']/df_all[f'fb_{age_min}_{age_max}_men']
    
df_all.to_csv(dropbox_path/'data/ins/imputed_final_all_ages_fb_data.csv',index=False)

In [ ]:
df_all=pd.read_csv(dropbox_path/'data/ins/imputed_final_all_ages_fb_data.csv')


# model fitting

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_absolute_error

def leave_one_column_out(column, data, outcome_var, indicator, model_type,X_const,y):
    recorder = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', column, 'pred', 'true', 'train_size'])
    for unique_value in data[column].unique():

        X_train_loo = X_const[data[column] != unique_value]
        y_train_loo = y[data[column] != unique_value]
        X_test_loo = X_const[data[column] == unique_value]
        y_test_loo = y[data[column] == unique_value]

        model = sm.OLS(y_train_loo, X_train_loo).fit()
        y_pred_loo = model.predict(X_test_loo).values

        for i in range(len(y_pred_loo)):

            recorder.loc[len(recorder)] = [f'{indicator} {model_type}', outcome_var, model_type, unique_value, y_pred_loo[i], y_test_loo.values[i], len(y_train_loo)]

    r2 = r2_score(recorder['true'], recorder['pred'])

    return recorder, r2

In [ ]:
import params

dropbox_fb_path  = params.dropbox_path.parents[2]/'pipeline'
fb_cols =["fb_18_999_men","fb_18_999_wom","fb_18_999_r"]
bg_cols = ['hdi','gdi','gdp_pcap','year']
model_specs = {'online': fb_cols, 
               'offline': bg_cols, 
               'combined_18_plus': fb_cols + bg_cols,
               'combined_18_plus_DHS_only': fb_cols + bg_cols}

cols = ['indicator','model_type','age_min','age_max','outcome_var','r2','adj_r2','loco_r2','mae','n']
df_res = pd.DataFrame(columns=cols)
#df_predictions = pd.DataFrame()
constructed_ages = [[25,49],[15,20],[21,49],[15,24],[25,34],[35,49]]

In [ ]:
for indicator in ['internet', 'mobile']:
    for model_type in ["combined_18_plus_DHS_only"]:

        # Load Facebook data, INS data, and outcome variables
        df_fb_all = pd.read_csv(dropbox_path/'data/ins/imputed_final_all_ages_fb_data.csv')
        df_ins = pd.read_csv(dropbox_path/'data/ins/imputed_final.csv')
        df_outcomes = pd.read_csv(dropbox_path/f'data/outcome_var/{indicator}_adolescent.csv')

        # Filter outcome data to include only DHS surveys if specified in the model type
        if 'DHS_only' in model_type:
            df_outcomes = df_outcomes.loc[df_outcomes['survey'].str.contains('dhs'),]

        # Add a 'year' column based on the survey start year
        df_outcomes['year'] = df_outcomes['survey_start']

        # Iterate through unique survey years to prepare background variables
        for survey_year in df_outcomes['year'].unique():
            # Load background data for the specific survey year
            df_background = pd.read_csv(dropbox_fb_path / f'model_fit/national/data/{indicator}/combined_multiple_years_no_missing_fb_aligned_{round(survey_year)}.csv')
            
            # Iterate through rows of the outcome data for the current survey year
            for ind, row in df_outcomes[df_outcomes['survey_start'] == survey_year].iterrows():
                # Skip rows for sanctioned countries
                if row['iso3'] not in params.sanction_countries + ['CUB', 'IRN', 'PRK', 'SYR', 'SDN', 'VEN']:
                    # Add background variables to the outcome data
                    for var in bg_cols:
                        if var != 'year':
                            df_outcomes.loc[ind, var] = df_background.loc[df_background['iso3'] == row['iso3'], var].values[0]

                    # Add INS variables if specified in the model type
                    if 'with_ins' in model_type:
                        for ages in ins_ages[model_type]:
                            for var_type in ["wom", "men", "r"]:
                                var_name = f'fb_{ages[0]}_{ages[1]}_{var_type}'
                                df_outcomes.loc[ind, f'ins_{ages[0]}_{ages[1]}_{var_type}'] = df_ins.loc[(df_ins['iso3'] == row['iso3']), var_name].values[0]

        # Merge Facebook data with outcome data
        df_outcomes = pd.merge(df_outcomes, df_fb_all[['iso3', 'year', 'fb_18_999_wom', 'fb_18_999_men', 'fb_18_999_r']], on=['iso3', 'year'], how='left')
        df_outcomes.to_csv(dropbox_path/f'data/{indicator}_adolescent_with_fb_data.csv',index=False)

In [ ]:
for indicator in ['internet', 'mobile']:
    for model_type in ["combined_18_plus_DHS_only"]:

        # Load Facebook data, INS data, and outcome variables
        df_fb_all = pd.read_csv(dropbox_path/'data/ins/imputed_final_all_ages_fb_data.csv')
        df_ins = pd.read_csv(dropbox_path/'data/ins/imputed_final.csv')
        df_outcomes = pd.read_csv(dropbox_path/f'data/outcome_var/{indicator}_adolescent.csv')

        # Filter outcome data to include only DHS surveys if specified in the model type
        if 'DHS_only' in model_type:
            df_outcomes = df_outcomes.loc[df_outcomes['survey'].str.contains('dhs'),]

        # Add a 'year' column based on the survey start year
        df_outcomes['year'] = df_outcomes['survey_start']

        # Iterate through unique survey years to prepare background variables
        for survey_year in df_outcomes['year'].unique():
            # Load background data for the specific survey year
            df_background = pd.read_csv(dropbox_fb_path / f'model_fit/national/data/{indicator}/combined_multiple_years_no_missing_fb_aligned_{round(survey_year)}.csv')
            
            # Iterate through rows of the outcome data for the current survey year
            for ind, row in df_outcomes[df_outcomes['survey_start'] == survey_year].iterrows():
                # Skip rows for sanctioned countries
                if row['iso3'] not in params.sanction_countries + ['CUB', 'IRN', 'PRK', 'SYR', 'SDN', 'VEN']:
                    # Add background variables to the outcome data
                    for var in bg_cols:
                        if var != 'year':
                            df_outcomes.loc[ind, var] = df_background.loc[df_background['iso3'] == row['iso3'], var].values[0]

                    # Add INS variables if specified in the model type
                    if 'with_ins' in model_type:
                        for ages in ins_ages[model_type]:
                            for var_type in ["wom", "men", "r"]:
                                var_name = f'fb_{ages[0]}_{ages[1]}_{var_type}'
                                df_outcomes.loc[ind, f'ins_{ages[0]}_{ages[1]}_{var_type}'] = df_ins.loc[(df_ins['iso3'] == row['iso3']), var_name].values[0]

        # Merge Facebook data with outcome data
        df_outcomes = pd.merge(df_outcomes, df_fb_all[['iso3', 'year', 'fb_18_999_wom', 'fb_18_999_men', 'fb_18_999_r']], on=['iso3', 'year'], how='left')
        
        # Iterate through constructed age groups to fit models
        for i in range(len(constructed_ages)):
            age_max, age_min = constructed_ages[i][1], constructed_ages[i][0]

            # Define model specifications based on the model type
            if 'combined_18_plus' in model_type:
                model_specs[model_type] = bg_cols + fb_cols
                model_types_to_iterate = [model_type]
            elif "with_ins" in model_type:
                model_specs[model_type + "_r"] = bg_cols + fb_cols + [f'ins_{ages[0]}_{ages[1]}_r' for ages in ins_ages[model_type.replace("_all", "").replace("_r", "")]]
                model_specs[model_type + "_all"] = bg_cols + fb_cols + [f'ins_{ages[0]}_{ages[1]}_{suffix}' for ages in ins_ages[model_type.replace("_all", "").replace("_r", "")] for suffix in ['r', 'wom', "men"]]
                model_types_to_iterate = [model_type + "_r", model_type + "_all"]

            # Iterate through model types and fit models
            for model_type_iterate in model_types_to_iterate:
                print(age_max, age_min, model_type_iterate, model_specs[model_type_iterate])
                for outcome_var in [f'{indicator}_fm_ratio_{age_min}_{age_max}', f'{indicator}_men_{age_min}_{age_max}', f'{indicator}_women_{age_min}_{age_max}']:
                    # Prepare data for model fitting
                    df_model = df_outcomes.dropna(subset=[outcome_var] + model_specs[model_type_iterate]).copy()
                    df_model['year'] = df_model['year'] - 2015  # Adjust year for modeling
                    X_const = add_constant(df_model[model_specs[model_type_iterate]])  # Add constant term for regression
                    model = OLS(df_model[outcome_var], X_const)  # Fit OLS model
                    results = model.fit()

                    # Save the fitted model
                    if not os.path.isdir(dropbox_path / f'model/{model_type_iterate}'):
                        os.makedirs(dropbox_path / f'model/{model_type_iterate}')
                    save_model(results, dropbox_path / f'model/{model_type_iterate}/{outcome_var}.pkl')

                    # Make predictions and clip values between 0 and 1
                    predictions = results.predict(add_constant(df_model[model_specs[model_type_iterate]]))
                    print(('range of predictions:', np.min(predictions), np.max(predictions)))
                    predictions = [np.min([x, 1]) for x in predictions]
                    predictions = [np.max([x, 0]) for x in predictions]

                    # Calculate mean absolute error (MAE)
                    mae = mean_absolute_error(df_model[outcome_var], predictions)

                    # Perform leave-one-column-out (LOCO) validation
                    recorder, loco_r2 = leave_one_column_out('iso3', df_model, outcome_var, indicator, model_type_iterate, X_const, df_model[outcome_var])

                    # Store model results
                    df_res = pd.concat([df_res, pd.DataFrame({
                        "indicator": indicator,
                        "model_type": model_type_iterate,
                        "age_min": age_min,
                        "age_max": age_max,
                        "outcome_var": outcome_var[0:-6].replace(f"{indicator}_", ''),
                        "r2": results.rsquared,
                        "adj_r2": results.rsquared_adj,
                        "loco_r2": loco_r2,
                        "n": results.nobs,
                        "mae": mae
                    }, index=[0])]).reset_index(drop=True)

                    # Clean up variables to free memory
                    del df_model, X_const, model, results, predictions, recorder

In [ ]:
df_res.to_csv(params.dropbox_data_path.parent/f'results/logs/ols_adolescent_models_results_summary.csv',index=False)

In [ ]:
# visualize the results with 2 by 2 plots facet by indicator and model type
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
# Create a new column for age groups in the results DataFrame
df_res['age_group'] = df_res.apply(lambda x: f"{x['age_min']}-{x['age_max']}", axis=1)
df_res['outcome_var'] = df_res['outcome_var'].replace({'fm_ratio': 'GGI',
                                                       'men': 'Men Level',
                                                       'women': 'Women Level'})
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

model_type_palette = {
    "Women Level": "#3a5b7e",
    "GGI": "#f0935d",
    "Men Level": "#74a4bc"
}

# Internet plot
sns.barplot(data=df_res.loc[df_res['indicator'] == 'internet'], x='age_group', y='loco_r2', hue='outcome_var', ax=axs[0], palette=model_type_palette)
axs[0].set_title('Internet-related outcomes',fontweight='bold')
axs[0].set_xlabel('Age Group')
axs[0].set_ylabel('Leave One Country Out R-squared')
axs[0].set_ylim(0, 1.05)
axs[0].legend().set_visible(False)

# Add text annotations for Internet plot
for container in axs[0].containers:
    axs[0].bar_label(container, fmt='%.2f', label_type='edge', fontsize=6)

# Mobile plot
sns.barplot(data=df_res.loc[df_res['indicator'] == 'mobile'], x='age_group', y='loco_r2', hue='outcome_var', ax=axs[1], palette=model_type_palette)
axs[1].set_title('Mobile-related outcomes', fontweight='bold')
axs[1].set_xlabel('Age Group')
axs[1].set_ylabel('')
axs[1].set_ylim(0, 1.05)
# remove y tick labels 
axs[1].set_yticklabels([])
legend = axs[1].legend(title='Outcome', loc='upper right')
for legend_handle in legend.get_patches():
    legend_handle.set_edgecolor('black')
    legend_handle.set_linewidth(1)

for ax in axs:
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(1)
# Add text annotations for Mobile plot
for container in axs[1].containers:
    axs[1].bar_label(container, fmt='%.2f', label_type='edge', fontsize=6)

# Remove top and right spines
for ax in axs:
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.tight_layout()
fig.suptitle('b. Adolescent Internet and Mobile Outcomes', fontweight='bold', fontsize=12,y=1.05,x=0.2)
plt.savefig(params.FIG / 'ols_loco_r2_by_modeltype_indicator_adolescent.pdf', bbox_inches='tight')

In [ ]:
df_res.to_csv(dropbox_path / f'result/{model_type}_model_fit_for_all_ages.csv', index=False)

# Predicted values in 2025 

In [ ]:
survey_year = 2025
model_type = 'combined_18_plus_DHS_only'
constructed_ages = [[15,20]]#[[25,49],[15,20],[21,49],[15,24],[25,34],[35,49]]
dropbox_fb_path  = params.dropbox_path.parents[2]/'pipeline'
fb_cols =["fb_18_999_men","fb_18_999_wom","fb_18_999_r"]
bg_cols = ['hdi','gdi','gdp_pcap','year']
model_specs = {'online': fb_cols, 
               'offline': bg_cols, 
               'combined_18_plus': fb_cols + bg_cols,
               'combined_18_plus_DHS_only': fb_cols + bg_cols}
# Prepare DataFrame for predictions
df_pred = pd.DataFrame(columns=['indicator','age_range','model_type','outcome_var','iso3','year','month','pred'])
df_background = pd.read_csv(dropbox_fb_path / f'model_fit/national/data/{indicator}/combined_multiple_years_no_missing_fb_aligned_{round(survey_year)}.csv')


In [ ]:

for indicator in ['internet', 'mobile']:
    
    # Load Facebook data, INS data, and outcome variables
    if 'ins' in model_type:
        # only load the INS data if the model type includes 'ins', and use the imputed fb data that contains all other younger ages 
        df_ins = pd.read_csv(dropbox_path/'data/ins/imputed_final.csv')
        ins_ages = {'combined_18_plus': [[15, 20], [21, 49], [15, 24], [25, 34], [35, 49]],
                    'combined_18_plus_DHS_only': [[15, 20], [21, 49], [15, 24], [25, 34], [35, 49]]}
        df_ins = df_ins.loc[df_ins['year'] == survey_year,]
        df_fb_all = pd.read_csv(dropbox_path/'data/ins/imputed_final_all_ages_fb_data.csv')
        
    else: 
        # read the latest Facebook data file that is used in the national pipeline analysis
        file = [x for x in os.listdir(dropbox_path.parents[1] / 'pipeline/preprocessed/national') if x.endswith('.csv')][0]
        df_fb_all = pd.read_csv(dropbox_path.parents[1] / f'pipeline/preprocessed/national/{file}')
        df_fb_all = df_fb_all.loc[df_fb_all['year'] == survey_year,]
        df_fb_all['fb_18_999_men'] = df_fb_all['fb_18_999_men']
        df_fb_all['fb_18_999_wom'] = df_fb_all['fb_18_999_wom']
    
    df_outcomes = df_fb_all[['iso3','year','month','fb_18_999_wom','fb_18_999_men','fb_18_999_r']].copy()



    if 'with_ins' in model_type:
        columns_to_combine = [f'ins_{ages[0]}_{ages[1]}_{var_type}' for ages in ins_ages[model_type.replace("_all","").replace("_r","")] for var_type in ["wom","men","r"]]
        df_outcomes = pd.merge(df_background,df_ins[['iso3']+columns_to_combine], on=['iso3'],how='left')


    #df_outcomes = pd.merge(df_background,df_fb_all[['iso3','year','fb_18_999_wom','fb_18_999_men','fb_18_999_r']], on=['iso3','year'],how='left').copy()
    #print(df_outcomes.columns)


    for i in range(len(constructed_ages)):
        age_max,age_min = constructed_ages[i][1],constructed_ages[i][0]
        age_range = f'{age_min}-{age_max}'
        for outcome_var in [f'{indicator}_fm_ratio_{age_min}_{age_max}',f'{indicator}_men_{age_min}_{age_max}',f'{indicator}_women_{age_min}_{age_max}']:

            model_spec = model_specs[model_type]
            for col in model_spec:
                if col not in df_outcomes.columns:
                    df_outcomes = pd.merge(df_outcomes, df_background[['iso3'] + [col]], on=['iso3'], how='left')
            
            model = load_model(dropbox_path/f'model/{model_type}/{outcome_var}.pkl')

            df_model = df_outcomes[['iso3','month']+model_spec].dropna(subset=model_spec).copy()
            df_model = df_model.drop_duplicates().reset_index(drop=True)
            df_model['year'] = df_model['year']-2015

            X_const = add_constant(df_model[model_spec])
            X_const['const'] = 1
            predictions = model.predict(X_const[model.params.index])

            print(age_range,outcome_var)

            df_model['pred'] = predictions

            temp_pred = df_model[['iso3','year','month','pred']].copy()
            temp_pred['year'] = temp_pred['year']+2015
            temp_pred['indicator']= indicator
            temp_pred['model_type']= model_type
            temp_pred['outcome_var'] = outcome_var
            temp_pred['age_range'] = age_range

            df_pred = pd.concat([df_pred,temp_pred])
            df_pred.reset_index(drop=True,inplace=True)

            print(f'prediction range {np.min(predictions)} to {np.max(predictions)}')
            #del temp_pred,df_model,X_const,model
            
df_pred['indicator'] = ['internet' if 'internet' in x else 'mobile' for x in df_pred['outcome_var']]
df_pred['pred'] = df_pred['pred'].clip(0, 1)  # Clip predictions between 0 and 1
df_pred = df_pred.loc[~df_pred['iso3'].isin(params.sanction_countries),]  # Filter for valid ISO3 codes


In [ ]:
df_pred.loc[df_pred['outcome_var'].str.contains('fm')].iso3.nunique()

## national adult predictions vs adolescent predictions

In [ ]:
df_adolescent_results = df_pred[['iso3','outcome_var','year','month','pred']].copy()
df_adolescent_results['outcome_var'] = [x.replace('_15_20','') for x in df_adolescent_results['outcome_var']]
df_adolescent_results.rename(columns={'pred': 'adolscent_predicted','outcome_var':'outcome'}, inplace=True)

In [ ]:
# now get the adults predictions for the same year 
files = os.listdir(dropbox_path.parents[1] / 'pipeline/result/national')
df_national_results = pd.concat([pd.read_csv(dropbox_path.parents[1] / f'pipeline/result/national/{file}') for file in files if file.endswith('.csv')], ignore_index=True)

df_national_results['year'] = df_national_results['date'].apply(lambda x: int(x.split('-')[0]))
df_national_results['month'] = df_national_results['date'].apply(lambda x: int(x.split('-')[1]))
df_national_results = df_national_results.loc[df_national_results['year'] == survey_year,]
df_national_results.rename(columns={'gid_0': 'iso3'}, inplace=True)
df_national_results.drop(columns =['Unnamed: 0', 'date'], inplace=True)

In [ ]:
df_results_all = pd.merge(df_adolescent_results, df_national_results, on=['iso3', 'year', 'month', 'outcome'], how='outer')
df_results_all = df_results_all.groupby(['iso3', 'year', 'outcome']).agg({
    'adolscent_predicted': 'mean',
    'predicted': 'mean'
}).reset_index()

In [ ]:
# get subregion from iso3
import utils
df_results_all= utils.mark_regions(df_results_all)
df_results_all['outcome'] = [x.replace('_',' ').title().replace("Fm",'Female-Male') for x in df_results_all['outcome']]

### plot facet by region

In [ ]:

custom_colors = {
    "Asia":"#e76254",  # Coral red
    "North America":"#7db782",  # Yellow-orange
    "Africa":"#f1975a",  # Light beige
    "South America":"#72c0c5",  # Pale cyan
    "Oceania":"#cf8dc5",  # Light blue
    "Europe":"#5f799a",  # Medium blue
    "Global":"#101c2c"   # Dark blue
}

In [ ]:

# visaulise the results, x-axis is the adult predicted, y-axis is the adolescent predicted,facet by outcome
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


custom_colors = [
    "#e76254",  # Coral red
    "#7db782",  # Yellow-orange
    "#f1975a",  # Light beige
    "#72c0c5",  # Pale cyan
    "#cf8dc5",  # Light blue
    "#5f799a",  # Medium blue
    "#101c2c"   # Dark blue
]


# Map custom colors to regions
region_colors = dict(zip(df_results_all['region'].unique(), custom_colors))

# Facet by outcome
hue_column = 'region'  # Change this to 'subregion' if you want to facet by subregion
g = sns.FacetGrid(df_results_all, col='outcome', hue='region', col_wrap=3, height=3, palette=region_colors,)
g.map(sns.scatterplot, 'predicted', 'adolscent_predicted', alpha=1, s=25)
g.set_axis_labels('Adult Predicted (15-49)', 'Adolescent Predicted (15-20)')

g.set_titles(col_template="{col_name}", fontweight='bold')
g.add_legend(title='Region', bbox_to_anchor=(0.08, -0.005), loc='upper left', ncol=3)
g.set(xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))

# Add a diagonal line for reference
for ax in g.axes.flat:
    ax.plot([-0.05, 1.05], [-0.05, 1.05], color='grey', linestyle='--', linewidth=1)
    ax.set_aspect('equal', adjustable='box')  # Ensure equal scaling of x and y axes
plt.savefig(params.FIG / "adolescent_adult_predictions_facet_by_region.pdf",bbox_inches='tight')

###  by continent

In [ ]:
df_results_all['outcome'] = [x.replace('Female-Male Ratio','GGI') for x in df_results_all['outcome']]

In [ ]:
 


# visaulise the results, x-axis is the adult predicted, y-axis is the adolescent predicted,facet by outcome
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


custom_colors = [
    
    "#7db782",  # Yellow-orange
    "#e76254",  # Coral red
    "#f1975a",  # Light beige
    "#5f799a",  # Medium blue
    "#72c0c5",  # Pale cyan
   
    "#cf8dc5",  # Light blue   # Dark blue
]




# Facet by outcome
hue_column = 'conti'  # Change this to 'subregion' if you want to facet by subregion

# Map custom colors to regions
region_colors = dict(zip(df_results_all[hue_column].unique(), custom_colors))


g = sns.FacetGrid(df_results_all, col='outcome', hue=hue_column, col_wrap=3, height=3, palette=region_colors,)
g.map(sns.scatterplot, 'predicted', 'adolscent_predicted', alpha=1, s=25)
g.set_axis_labels('Adult Predicted (15-49)', 'Adolescent Predicted (15-20)')

g.set_titles(col_template="{col_name}", fontweight='bold')
g.add_legend(title='Continent', bbox_to_anchor=(0.18, 0), loc='upper left', ncol=3)
g.set(xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))

# Add a diagonal line for reference
for ax in g.axes.flat:
    ax.plot([-0.05, 1.05], [-0.05, 1.05], color='grey', linestyle='--', linewidth=1)
    ax.set_aspect('equal', adjustable='box')  # Ensure equal scaling of x and y axes
plt.savefig(params.FIG / "adolescent_adult_predictions_facet_by_continent.pdf",bbox_inches='tight')


### plot facet by HDI 

In [ ]:
df_results_all_hdi = pd.merge(df_results_all, df_background.loc[df_background['align_year'] == survey_year,['iso3','hdi','gdi','gdp_pcap']], on='iso3', how='left')

In [ ]:
values_dict = {key: value for key, value in df_results_all_hdi['hdi'].describe().items()}
values_dict

In [ ]:
df_results_all_hdi['hdi_bins'] = pd.cut(df_results_all_hdi['hdi'], bins=[values_dict['min'], values_dict['25%'], values_dict['50%'], values_dict['75%'], values_dict['max']], labels=['Low', 'Lower-Middle', 'Upper-Middle', 'High'], include_lowest=True)

In [ ]:
# Visualize the results, x-axis is the adult predicted, y-axis is the adolescent predicted, facet by outcome
import seaborn as sns
import matplotlib.pyplot as plt

# Facet by outcome
hue_column = 'hdi_bins'  # Change this to 'subregion' if you want to facet by subregion


custom_colors = [
    "#5f799a",  # Medium blue
    
    
    "#72c0c5",  # Pale cyan
    "#f1975a",  # Light beige
    "#e76254",  # Coral red

    

]

# Map custom colors to regions
region_colors = dict(zip(['High', 'Upper-Middle','Lower-Middle','Low'], custom_colors)) #df_results_all_hdi[hue_column].unique()

g = sns.FacetGrid(df_results_all_hdi, col='outcome', hue=hue_column, col_wrap=3, height=3,palette=region_colors,)
g.map(sns.scatterplot, 'predicted', 'adolscent_predicted', alpha=1, s=20)
g.set_axis_labels('Adult Predicted (15-49)', 'Adolescent Predicted (15-20)')

g.set_titles(col_template="{col_name}", fontweight='bold')
g.add_legend(title='HDI Category', bbox_to_anchor=(0.17, -0.005), loc='upper left', ncol=4)
g.set(xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))

# Add a diagonal line for reference
for ax in g.axes.flat:
    ax.plot([-0.05, 1.05], [-0.05, 1.05], color='grey', linestyle='--', linewidth=1)
    ax.set_aspect('equal', adjustable='box')  # Ensure equal scaling of x and y axes


# map view? 

In [ ]:
import numpy as np
import utils
import matplotlib as mpl
import params
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# Download world country boundaries (Natural Earth, GeoJSON format)
world_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
world = gpd.read_file(world_url)

# Check ISO code column name
world = world.rename(columns={'ISO_A3': 'gid_0'})  # Make sure this matches your df's 'gid_0'
import matplotlib.colors as mcolors

# Custom red-dominant colormap
colors = [
    "#990000",  # deep red
    "#d7301f",  # brick red
    "#ef6548",  # orange-red
    "#fc8d59",  # salmon
    "#fdbb84",  # peach
    "#fef0d9",  # light peach / near-white
    "#d9f0f9",  # very light blue
    "#91bfdb",  # light blue
    "#4575b4",  # medium blue
    "#313695"   # deep blue
]
cmap = mcolors.LinearSegmentedColormap.from_list("custom_red", colors)


In [ ]:
indicator = 'internet'
age_range = '15_24'
outcome_var = f"{indicator}_women_{age_range}"

# Combined model fit graph (adult and adolescent)

In [ ]:
# read results 
deleting_itu = True
with_year = True
results_df=pd.read_csv(params.dropbox_data_path.parent/f'results/logs/ols{"_no_ITU" if deleting_itu else ""}{"_with_year" if with_year else ""}_results_summary.csv')
results_df_CIS=results_df.loc[results_df['model_type'].str.contains('CIS')]

df_res= pd.read_csv(params.dropbox_data_path.parent/f'results/logs/ols_adolescent_models_results_summary.csv')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path


title_size = 10 
overall_title_size = 12
annotation_size = 9
# =====================
# Prep: Adolescents
# =====================
df_adol = df_res.copy()
df_adol['age_group'] = df_adol.apply(lambda x: f"{x['age_min']}-{x['age_max']}", axis=1)

# Explicitly set categorical order for plotting
ordered_age_groups = (
    df_adol['age_group']
    .drop_duplicates()
    .tolist()
)
ordered_age_groups = sorted(ordered_age_groups, key=lambda x: int(x.split('-')[0]))
df_adol['age_group'] = pd.Categorical(df_adol['age_group'], categories=ordered_age_groups, ordered=True)

df_adol['outcome_var'] = df_adol['outcome_var'].replace({
    'fm_ratio': 'GGI',
    'men': 'Men Level',
    'women': 'Women Level'
})

adol_palette = {
    "Women Level": "#3a5b7e",
    "GGI": "#f0935d",
    "Men Level": "#74a4bc"
}

# =====================
# Prep: Adults
# =====================
df_adult = results_df_CIS.copy()
df_adult['model_type'] = df_adult['model_type'].str.replace('_with_CIS', '', regex=False)

# Nice x labels
df_adult["formatted_label"] = (
    df_adult["outcome_var"]
    .str.capitalize()
    .str.replace("_", "\n", regex=False)
    .str.replace("men", "Men level", regex=False)
    .str.replace("wom", "Women level", regex=False)
    .str.replace("ggi", "GGI", regex=False)
)

adult_internet = df_adult[df_adult["outcome_var"].str.contains("internet", case=False)]
adult_mobile   = df_adult[df_adult["outcome_var"].str.contains("mobile",   case=False)]

adult_palette = {
    "combined": "#3a5b7e",
    "online":   "#f0935d",
    "offline":  "#74a4bc"
}

# =====================
# Plot: one 2x2 figure
# =====================
fig, axs = plt.subplots(2, 2, figsize=(10, 8), sharey=True)

# ---- A1: Adults — Internet
ax = axs[0, 0]
sns.barplot(
    data=adult_internet,
    x="formatted_label", y="loco_r2", hue="model_type",
    palette=adult_palette, ax=ax, saturation=1
)
ax.set_title("Internet-related outcomes", fontweight='bold', fontsize=title_size)
ax.set_xlabel("")
ax.set_ylabel("Leave-one-country-out  R²")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
for p in ax.patches: p.set_edgecolor("black"); p.set_linewidth(1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", label_type="edge", fontsize=annotation_size)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.legend_.remove()

# ---- A2: Adults — Mobile
ax = axs[0, 1]
sns.barplot(
    data=adult_mobile,
    x="formatted_label", y="loco_r2", hue="model_type",
    palette=adult_palette, ax=ax, saturation=1
)
ax.set_title("Mobile-related outcomes", fontweight='bold', fontsize=title_size)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_ylim(0, 1.05)
ax.set_yticklabels([])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
for p in ax.patches: p.set_edgecolor("black"); p.set_linewidth(1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", label_type="edge", fontsize=annotation_size)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
# Adult legend (right-top)
adult_legend_elems = [
    Patch(facecolor=adult_palette[m], edgecolor='black', linewidth=1.2, label=m.capitalize())
    for m in ['online', 'offline', 'combined']
]
ax.legend(title='Model Type', handles=adult_legend_elems, loc='upper right')

# ---- B1: Adolescents — Internet
ax = axs[1, 0]
sns.barplot(
    data=df_adol.loc[df_adol['indicator'] == 'internet'],
    x='age_group', y='loco_r2', hue='outcome_var',
    palette=adol_palette, ax=ax, saturation=1
)
ax.set_title("Internet-related outcomes", fontweight='bold', fontsize=title_size)
ax.set_xlabel("Age Group")
ax.set_ylabel("Leave-one-country-out  R²")
ax.set_ylim(0, 1.05)
for p in ax.patches: p.set_edgecolor("black"); p.set_linewidth(1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", label_type="edge", fontsize=annotation_size-2)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.legend_.remove()

# ---- B2: Adolescents — Mobile
ax = axs[1, 1]
sns.barplot(
    data=df_adol.loc[df_adol['indicator'] == 'mobile'],
    x='age_group', y='loco_r2', hue='outcome_var',
    palette=adol_palette, ax=ax, saturation=1
    )
ax.set_title("Mobile-related outcomes", fontweight='bold', fontsize=title_size)
ax.set_xlabel("Age Group")
ax.set_ylabel("")
ax.set_ylim(0, 1.05)
ax.set_yticklabels([])
for p in ax.patches: p.set_edgecolor("black"); p.set_linewidth(1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", label_type="edge", fontsize=annotation_size-2)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
# Adolescent legend (right-bottom)
adol_legend_elems = [
    Patch(facecolor=adol_palette[m], edgecolor='black', linewidth=1.2, label=m)
    for m in ['Women Level', 'GGI', 'Men Level']
]
ax.legend(title='Outcome', handles=adol_legend_elems, loc='upper right')

# ---- Row labels & layout
# Panel labels like your originals
axs[0,0].text(-0.15, 1.15, "A. Adult Internet and Mobile Outcomes \n", transform=axs[0,0].transAxes, fontsize=overall_title_size, fontweight='bold', va='top')
axs[1,0].text(-0.15, 1.15, "B. Adolescent Internet and Mobile Outcomes\n", transform=axs[1,0].transAxes, fontsize=overall_title_size, fontweight='bold', va='top')

plt.tight_layout()
plt.subplots_adjust(top=0.90)

# Save
outpath = params.FIG / 'ols_loco_r2_combined.pdf'
plt.savefig(outpath, bbox_inches='tight')
print(f"Saved: {outpath}")

## model coefficients 

In [ ]:
from statsmodels.iolib.summary2 import summary_col
pd.options.display.max_rows = 4000

# load statistics 
all_models = {}
model_type_iterate = 'combined_18_plus_DHS_only'
for indicator in ['internet', 'mobile']:
    all_models[indicator] = []
    for outcome_var in [f'{indicator}_fm_ratio_15_20', f'{indicator}_women_15_20', f'{indicator}_men_15_20']:
        for model_type in ['combined_with_CIS']:
            model_filename = dropbox_path / f'model/{model_type_iterate}/{outcome_var}.pkl'
            model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
            model = load_model(model_filepath)
            all_models[indicator].append(model)

In [ ]:
# internet results in latex format
print(summary_col(all_models['internet']+all_models['mobile'],stars=True,float_format='%0.3f').as_latex())

In [ ]:
model_filename